In [ ]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from collections import Counter
from autokmc.graph import build_graph

In [ ]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../../.."))

In [ ]:
### Load the Allegro/NequIP calculator
import torch
from nequip.ase import NequIPCalculator

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_MODEL_FILE = "asehcocuau.nequip.pt2" if _DEVICE == "cuda" else "cpuhcocuau.nequip.pth"
_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", _MODEL_FILE)
print(f"Using device : {_DEVICE}")
print(f"Model file   : {_MODEL_FILE}")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device=_DEVICE,
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

In [ ]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

In [ ]:
## Visualise the slab
view_x3d(slab)

In [ ]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

In [ ]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

In [ ]:
### Build the graph for the slab
graph = build_graph(slab)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

In [ ]:
### Find all adsorption sites for H, C, and O
from autokmc.default_sites import find_sites_for_element, reduce_sites_by_isomorphism

for element in ["H", "C", "O"]:
    find_sites_for_element(graph, element, verbose=True)
    print()

In [ ]:
### Reduce to unique iso-classes at shell depths 1, 2, 3
for element in ["H", "C", "O"]:
    print(f"\n{'='*50}")
    print(f"  {element}")
    print(f"{'='*50}")
    for n_shells in [1, 2, 3]:
        reduce_sites_by_isomorphism(graph, element, n_shells=n_shells, verbose=True)
        print()

In [ ]:
### Optimise adsorbate positions for all sites
from autokmc.default_sites import optimise_site_positions

for element in ["H", "C", "O"]:
    optimise_site_positions(graph, element, verbose=True)
    print()

In [ ]:
### Plotly helper: plot initial centroids vs optimised positions
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict

_ELEM_COLOURS = {
    "Cu": "#B87333", "Au": "#FFD700", "Ag": "#C0C0C0",
    "H":  "#E0E0E0", "C":  "#404040", "O":  "#FF4444",
    "N":  "#4488FF", "S":  "#DDDD00",
}
_SITE_COLOURS = {1: "#FF6B6B", 2: "#4ECDC4", 3: "#45B7D1", 4: "#96CEB4"}
_SITE_LABELS  = {1: "top", 2: "bridge", 3: "hollow", 4: "4-fold"}


def _raw_centroids(g, element):
    """MIC-aware (or Euclidean for nanoparticles) centroid per raw site."""
    _cell = np.array(g.graph["cell"])
    _pbc  = np.asarray(g.graph.get("pbc", [True, True, False]), dtype=bool)
    _use_mic = _pbc.any()
    if _use_mic:
        try:
            _cell_inv = np.linalg.inv(_cell)
        except np.linalg.LinAlgError:
            _use_mic = False
    out = {}
    for k, cliques in g.graph["sites"][element].items():
        pts = []
        for clique in cliques:
            bp  = np.array([g.nodes[n]["position"] for n in clique])
            ref = bp[0]
            dv  = bp - ref
            if _use_mic:
                frac = dv @ _cell_inv
                for i in range(3):
                    if _pbc[i]:
                        frac[:, i] -= np.round(frac[:, i])
                mic_rel = frac @ _cell
            else:
                mic_rel = dv
            pts.append(ref + mic_rel.mean(axis=0))
        out[k] = pts
    return out


def _atom_traces(atoms, showlegend=True):
    by_sym = defaultdict(list)
    for atom in atoms:
        by_sym[atom.symbol].append(atom.position)
    traces = []
    for sym, positions in sorted(by_sym.items()):
        arr = np.array(positions)
        traces.append(go.Scatter3d(
            x=arr[:, 0], y=arr[:, 1], z=arr[:, 2],
            mode="markers", name=sym,
            legendgroup=sym, showlegend=showlegend,
            marker=dict(size=5, color=_ELEM_COLOURS.get(sym, "#AAAAAA"),
                        opacity=0.7, line=dict(width=0.5, color="rgba(0,0,0,0.4)")),
        ))
    return traces


def _site_traces(positions_by_k, showlegend=True):
    traces = []
    for k, pts in sorted(positions_by_k.items()):
        if not pts:
            continue
        arr = np.array(pts)
        traces.append(go.Scatter3d(
            x=arr[:, 0], y=arr[:, 1], z=arr[:, 2],
            mode="markers", name=_SITE_LABELS.get(k, f"{k}-fold"),
            legendgroup=f"site_k{k}", showlegend=showlegend,
            marker=dict(size=6, color=_SITE_COLOURS.get(k, "#DDDDDD"),
                        symbol="diamond", opacity=0.95,
                        line=dict(width=1, color="black")),
        ))
    return traces


def plot_sites(g, element, atoms):
    """Side-by-side Plotly figure: initial centroids (left) vs optimised (right)."""
    initial   = _raw_centroids(g, element)
    optimised = g.graph.get("site_positions", {}).get(element, {})

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"{element}  –  initial site centroids",
                        f"{element}  –  optimised positions"),
        specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
        horizontal_spacing=0.02,
    )
    for col, (site_pos, show_leg) in enumerate(
            [(initial, True), (optimised, False)], start=1):
        for t in _atom_traces(atoms, show_leg):
            fig.add_trace(t, row=1, col=col)
        for t in _site_traces(site_pos, show_leg):
            fig.add_trace(t, row=1, col=col)

    camera     = dict(eye=dict(x=0, y=-1.8, z=1.2))
    axis_style = dict(showbackground=False, showgrid=True,
                      zeroline=False, showticklabels=False)
    scene_cfg  = dict(aspectmode="data", camera=camera,
                      xaxis=axis_style, yaxis=axis_style, zaxis=axis_style)
    fig.update_layout(
        title=dict(text=f"Adsorption sites: {element}", x=0.5),
        height=620, width=1300,
        scene=scene_cfg, scene2=scene_cfg,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig.show()

In [ ]:
### Visualise initial vs optimised site positions for H, C, O
for _elem in ["H", "C", "O"]:
    plot_sites(graph, _elem, slab)

In [ ]:
# =============================================================================
#  NANOPARTICLE EXAMPLE
# =============================================================================

In [ ]:
### Build a small Cu nanoparticle (~300 atoms)
nanoparticle = build_nanoparticle(
    composition="Cu",
    crystal_structure="fcc",
    surface_energies={(1, 1, 1): 1.10, (1, 0, 0): 1.29, (1, 1, 0): 1.51},
    target_atoms=300,
    calculator=make_calc(),
    verbose=True,
)

print(f"\nNanoparticle formula : {nanoparticle.get_chemical_formula()}")
print(f"Nanoparticle atoms   : {len(nanoparticle)}")
pos_np = nanoparticle.get_positions()
r_max  = np.linalg.norm(pos_np - pos_np.mean(axis=0), axis=1).max()
print(f"Diameter             : {2*r_max:.2f} Å  ({2*r_max/10:.2f} nm)")

In [ ]:
### Find surface atoms on the nanoparticle (convex-hull method)
# find_surface_atoms auto-detects the geometry; for a non-periodic nanoparticle
# it returns (surface_mask, surface_indices, hull, method) — four values.
np_surface_mask, np_surface_indices, np_hull, np_method = find_surface_atoms(
    nanoparticle,
    tag_atoms=True,   # writes nanoparticle.arrays["surface"]
)

print(f"Detection method : {np_method}")
print(f"Surface atoms    : {np_surface_mask.sum()} / {len(nanoparticle)}")

In [ ]:
### Build the graph for the nanoparticle
np_graph = build_graph(nanoparticle)
print(f"Graph has {np_graph.number_of_nodes()} nodes and {np_graph.number_of_edges()} edges.")
np_type_counts = Counter(d["type"] for _, d in np_graph.nodes(data=True))
for t, n in sorted(np_type_counts.items()):
    print(f"  {t:10s} : {n}")

In [ ]:
### Find adsorption sites on the nanoparticle for H, C, and O
for element in ["H", "C", "O"]:
    find_sites_for_element(np_graph, element, verbose=True)
    print()

In [ ]:
### Reduce to unique iso-classes (n_shells = 1, 2)
for element in ["H", "C", "O"]:
    print(f"\n{'='*50}")
    print(f"  {element}")
    print(f"{'='*50}")
    for n_shells in [1, 2]:
        reduce_sites_by_isomorphism(np_graph, element, n_shells=n_shells, verbose=True)
        print()

In [ ]:
### Optimise adsorbate positions on the nanoparticle
for element in ["H", "C", "O"]:
    optimise_site_positions(np_graph, element, verbose=True)
    print()

In [ ]:
### Visualise nanoparticle sites: initial centroids vs optimised positions
for _elem in ["H", "C", "O"]:
    plot_sites(np_graph, _elem, nanoparticle)

In [ ]:
# =============================================================================
#  Cu(100) SURFACE
# =============================================================================

In [ ]:
### Build a Cu(100) surface slab
slab_100 = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 0, 0),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True,
)

print(f"\nSlab formula : {slab_100.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab_100)}")
cell_100 = slab_100.get_cell()
print(f"Cell (Å)     : a={cell_100[0,0]:.3f}  b={cell_100[1,1]:.3f}  c={cell_100[2,2]:.3f}")

In [ ]:
### Surface atoms, graph, sites — Cu(100)
surf_mask_100, surf_idx_100, method_100 = find_surface_atoms(
    slab_100, which="top", tag_atoms=True)
print(f"Method: {method_100}  |  surface atoms: {surf_mask_100.sum()} / {len(slab_100)}")

graph_100 = build_graph(slab_100)
print(f"Graph: {graph_100.number_of_nodes()} nodes, {graph_100.number_of_edges()} edges")
type_counts_100 = Counter(d["type"] for _, d in graph_100.nodes(data=True))
for t, n in sorted(type_counts_100.items()):
    print(f"  {t:10s} : {n}")

for element in ["H", "C", "O"]:
    find_sites_for_element(graph_100, element, verbose=True)
    for n_shells in [1, 2]:
        reduce_sites_by_isomorphism(graph_100, element, n_shells=n_shells, verbose=True)
    optimise_site_positions(graph_100, element, verbose=True)
    print()

In [ ]:
### Visualise Cu(100) sites
for _elem in ["H", "C", "O"]:
    plot_sites(graph_100, _elem, slab_100)

In [ ]:
# =============================================================================
#  Cu(211) SURFACE  (stepped)
# =============================================================================

In [ ]:
### Build a Cu(211) surface slab
slab_211 = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(2, 1, 1),
    calculator=make_calc(),
    min_slab_size=10.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True,
)

print(f"\nSlab formula : {slab_211.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab_211)}")
cell_211 = slab_211.get_cell()
print(f"Cell (Å)     : a={cell_211[0,0]:.3f}  b={cell_211[1,1]:.3f}  c={cell_211[2,2]:.3f}")

In [ ]:
### Surface atoms, graph, sites — Cu(211)
surf_mask_211, surf_idx_211, method_211 = find_surface_atoms(
    slab_211, which="top", tag_atoms=True)
print(f"Method: {method_211}  |  surface atoms: {surf_mask_211.sum()} / {len(slab_211)}")

graph_211 = build_graph(slab_211)
print(f"Graph: {graph_211.number_of_nodes()} nodes, {graph_211.number_of_edges()} edges")
type_counts_211 = Counter(d["type"] for _, d in graph_211.nodes(data=True))
for t, n in sorted(type_counts_211.items()):
    print(f"  {t:10s} : {n}")

for element in ["H", "C", "O"]:
    find_sites_for_element(graph_211, element, verbose=True)
    for n_shells in [1, 2]:
        reduce_sites_by_isomorphism(graph_211, element, n_shells=n_shells, verbose=True)
    optimise_site_positions(graph_211, element, verbose=True)
    print()

In [ ]:
### Visualise Cu(211) sites
for _elem in ["H", "C", "O"]:
    plot_sites(graph_211, _elem, slab_211)

In [ ]:
# =============================================================================
#  SCALING BENCHMARK  —  Cu(111), element C
#  Measures wall time for find_sites + reduce_sites_by_isomorphism (n_shells=2)
#  as the slab lateral size increases.
# =============================================================================

In [ ]:
import time
import plotly.express as px
import pandas as pd

_BENCH_SIZES = [6.0, 9.0, 12.0, 15.0, 18.0, 22.0]   # goal_x = goal_y  (Å)
_BENCH_ELEM  = "C"
_BENCH_SHELLS = 2

bench_rows = []
print(f"{'size_Å':>8}  {'n_atoms':>7}  {'n_surf':>7}  "
      f"{'n_sites':>8}  {'t_find':>8}  {'t_reduce':>9}  {'t_opt':>7}")
print("-" * 70)

for _goal in _BENCH_SIZES:
    # ── build slab ────────────────────────────────────────────────────────
    _slab = build_surface(
        composition="Cu", crystal_structure="fcc", miller_index=(1,1,1),
        calculator=make_calc(),
        min_slab_size=8.0, min_vacuum_size=12.0,
        goal_x=_goal, goal_y=_goal,
        n_freeze_layers=2, verbose=False, orthogonalise=True,
    )
    find_surface_atoms(_slab, which="top", tag_atoms=True)
    _g = build_graph(_slab)
    _n_surf = sum(1 for _, d in _g.nodes(data=True) if d["type"] == "surface")

    # ── time find_sites ───────────────────────────────────────────────────
    _t0 = time.perf_counter()
    find_sites_for_element(_g, _BENCH_ELEM, verbose=False)
    _t_find = time.perf_counter() - _t0
    _n_sites = sum(len(v) for v in _g.graph["sites"][_BENCH_ELEM].values())

    # ── time reduce ───────────────────────────────────────────────────────
    _t0 = time.perf_counter()
    reduce_sites_by_isomorphism(_g, _BENCH_ELEM, n_shells=_BENCH_SHELLS, verbose=False)
    _t_reduce = time.perf_counter() - _t0

    # ── time optimise ─────────────────────────────────────────────────────
    _t0 = time.perf_counter()
    optimise_site_positions(_g, _BENCH_ELEM, verbose=False)
    _t_opt = time.perf_counter() - _t0

    bench_rows.append({
        "goal_Å"  : _goal,
        "n_atoms" : len(_slab),
        "n_surf"  : _n_surf,
        "n_sites" : _n_sites,
        "t_find"  : _t_find,
        "t_reduce": _t_reduce,
        "t_opt"   : _t_opt,
        "t_total" : _t_find + _t_reduce + _t_opt,
    })
    print(f"{_goal:>8.1f}  {len(_slab):>7}  {_n_surf:>7}  "
          f"{_n_sites:>8}  {_t_find:>7.2f}s  {_t_reduce:>8.2f}s  {_t_opt:>6.2f}s")

bench_df = pd.DataFrame(bench_rows)
print("\nDone.")

In [ ]:
### Plot benchmark results
fig_bench = px.line(
    bench_df, x="n_surf", y=["t_find", "t_reduce", "t_opt", "t_total"],
    markers=True,
    labels={"n_surf": "Surface atoms", "value": "Wall time (s)", "variable": "Step"},
    title=f"Cu(111) site-classification scaling — element {_BENCH_ELEM}, n_shells={_BENCH_SHELLS}",
)
fig_bench.update_layout(height=450, width=800)
fig_bench.show()

# Also show a secondary x-axis with total atom count
fig_bench2 = px.line(
    bench_df, x="n_atoms", y="t_total",
    markers=True,
    labels={"n_atoms": "Total atoms in slab", "t_total": "Total wall time (s)"},
    title=f"Total time vs slab size  ({_BENCH_ELEM}, n_shells={_BENCH_SHELLS})",
)
fig_bench2.update_layout(height=400, width=700)
fig_bench2.show()